In [2]:
import re
import json
import numpy as np
from collections import Counter

In [4]:
# config
CAPTION_FILE = "../data/archive/captions.txt"
VOCAB_FILE = "../data/vocab/vocab.json"
SEQUENCE_FILE = "../data/vocab/caption_sequences.npy"
SPECIAL_TOKENS = [
    "<pad>",
    "<start>",
    "<end>"
]

In [5]:
# load

captions = []
with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

    lines = lines[1:] # header jangan masuk

    for line in lines:
        line = line.strip()
        if not line:
            continue

        image_name, caption = line.split(",", 1)
        captions.append(caption)

print("Total captions:", len(captions))


Total captions: 40455


In [6]:
# cleaning

cleaned_captions = []

for caption in captions:
    caption = caption.lower() #lowercase
    caption = re.sub(r"[^a-z0-9\s]", "", caption) # hapus tanda baca
    caption = re.sub(r"\s+", " ", caption).strip() # hapus spasi  berlebih
    caption = f"<start> {caption} <end>" # token
    cleaned_captions.append(caption)

print("\nSample cleaned caption:")
print(cleaned_captions[0])



Sample cleaned caption:
<start> a child in a pink dress is climbing up a set of stairs in an entry way <end>


In [7]:
# vocabulary
word_counter = Counter()
for caption in cleaned_captions:
    tokens = caption.split() # tokenisasi
    word_counter.update(tokens)

# vocabulary dictionary
vocab = {}

# untuk special tokens
for idx, token in enumerate(SPECIAL_TOKENS):
    vocab[token] = idx

current_idx = len(vocab)
for word in word_counter:
    if word not in vocab:
        vocab[word] = current_idx
        current_idx += 1

print("\nVocabulary size:", len(vocab))


Vocabulary size: 8831


In [8]:
# save
with open(VOCAB_FILE, "w", encoding="utf-8") as f:
    json.dump(vocab, f, indent=2)

print("\nVocabulary saved to:", VOCAB_FILE)


Vocabulary saved to: ../data/vocab/vocab.json


In [9]:
# caption

caption_sequences = []
for caption in cleaned_captions:
    tokens = caption.split()
    sequence = [
        vocab[token]
        for token in tokens
    ]
    caption_sequences.append(sequence)

print("\nSample sequence:")
print(caption_sequences[0])


Sample sequence:
[1, 3, 4, 5, 3, 6, 7, 8, 9, 10, 3, 11, 12, 13, 5, 14, 15, 16, 2]


In [10]:
# padding
max_length = max(len(seq) for seq in caption_sequences)
print("\nMaximum sequence length:", max_length)

pad_value = vocab["<pad>"]
padded_sequences = []

for seq in caption_sequences:
    padding_needed = max_length - len(seq)
    padded_seq = np.pad(
        seq,
        pad_width=(0, padding_needed),
        mode="constant",
        constant_values=pad_value
    )
    padded_sequences.append(padded_seq)

padded_sequences = np.array(
    padded_sequences,
    dtype=np.int32
)

print("\nPadded sequences shape:")
print(padded_sequences.shape)


Maximum sequence length: 38

Padded sequences shape:
(40455, 38)


In [11]:
# save
np.save(SEQUENCE_FILE, padded_sequences)
print("\nSequences saved to:", SEQUENCE_FILE)


Sequences saved to: ../data/vocab/caption_sequences.npy


In [ ]:
# checking just in case
sample_sequence = caption_sequences[0]
input_sequence = sample_sequence[:-1]
target_sequence = sample_sequence[1:]

print("\nInput sequence:")
print(input_sequence)

print("\nTarget sequence:")
print(target_sequence)


Input sequence:
[1, 3, 4, 5, 3, 6, 7, 8, 9, 10, 3, 11, 12, 13, 5, 14, 15, 16]

Target sequence:
[3, 4, 5, 3, 6, 7, 8, 9, 10, 3, 11, 12, 13, 5, 14, 15, 16, 2]
